In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print("Current working directory:", os.getcwd())

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from mech_analysis.patch_edge import get_top_edges, get_top_contrast_edges_by_subtraction, get_attention_hooks, SpanEdge
from mech_analysis.evaluate import get_attribution_results, patch_evaluate
from mech_analysis.pattern_cache import get_attention_pattern, cache_attention_patterns
from model_utils.base_model import BaseModel
from utils.data_util import get_data_list


In [ ]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# model_name = "qwen3-4b"
# model_name = "watt-tool-8b"
model_name = "toolace-2.5-8b"

base_model = BaseModel.create(model_name)
model = base_model.model

In [ ]:
add = [1]
split = "test"
trunc = 500

sem_data, cf_sem_data = get_data_list(base_model, add, "semantic", split, trunc=trunc)
str_data, cf_str_data = get_data_list(base_model, add, "structural", split, trunc=trunc)

In [ ]:
sem_attr_results = get_attribution_results(model_name, add, trunc, "train", "semantic")
str_attr_results = get_attribution_results(model_name, add, trunc, "train", "structural")


In [ ]:
sem_attr = sem_attr_results["attr"]["total"]
cf_sem_attr = sem_attr_results["cf_attr"]["total"]
str_attr = str_attr_results["attr"]["total"]
cf_str_attr = str_attr_results["cf_attr"]["total"]

span_types = sem_attr_results["span_types"]


In [ ]:
totoal_edge = int(sem_attr.shape[0] * sem_attr.shape[1] * sem_attr.shape[2] * (sem_attr.shape[3] + 1) / 2)
print(totoal_edge)
top_sem_c_edges = get_top_contrast_edges_by_subtraction(sem_attr, cf_sem_attr, span_types, top_n=totoal_edge)
top_str_c_edges = get_top_contrast_edges_by_subtraction(str_attr, cf_str_attr, span_types, top_n=totoal_edge)

In [ ]:
top_str_c_edges[0]

In [ ]:
import matplotlib as mpl
mpl.rcParams['path.simplify'] = False
mpl.rcParams['path.simplify_threshold'] = 0.0
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys
import numpy as np

# Color palette (cycled)
colors_list = [
    "#9E9E9E", "#F0988C", "#B883D4",
    "#A1A9D0", "#F6CAE5", "#96CCCB",
    "#CFEAF1",
]

# Edge dataclass
class SpanEdge:
    def __init__(self, layer, head, src_span_name, dst_span_name, score):
        self.layer = layer
        self.head = head
        self.src_span_name = src_span_name
        self.dst_span_name = dst_span_name
        self.score = score

def darken_color(color, factor=0.75):
    """
    factor < 1 -> darker
    """
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)

def deepen_color(color, v_factor=0.85, s_factor=1.25):
    """
    v_factor < 1 -> darker
    s_factor > 1 -> more saturated
    """
    r, g, b = mcolors.to_rgb(color)
    h, s, v = colorsys.rgb_to_hsv(r, g, b)

    s = min(1.0, s * s_factor)
    v = max(0.0, v * v_factor)

    r2, g2, b2 = colorsys.hsv_to_rgb(h, s, v)
    return (r2, g2, b2)

# Sigmoid curve path
def sigmoid_path(x_start, y_start, x_end, y_end):
    t = np.linspace(0, 1, 800)
    sig = 1 / (1 + np.exp(-12 * (t - 0.5)))
    x = x_start + (x_end - x_start) * t
    y = y_start + (y_end - y_start) * sig
    return x, y


# Main plotting function
def visualize_custom_final(
    edges,
    all_spans,
    k=20,
    figsize=(15, 6),
    label_interval=2,
    title="Attention Flow",
    span_rename_map=None,
    span_color_map=None,   # shared across paired plots
):
    # Args
    if span_rename_map is None:
        span_rename_map = {}

    if span_color_map is None:
        span_color_map = {}

    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

    # Select top-K edges
    sorted_edges = sorted(edges, key=lambda x: x.score, reverse=True)
    top_edges = sorted_edges[:k]
    if not top_edges:
        return

    # Active layers and mapping
    active_layers = set()
    for e in top_edges:
        active_layers.add(e.layer)
        active_layers.add(e.layer - 1)

    sorted_active_layers = sorted(active_layers)
    layer_map = {real: idx for idx, real in enumerate(sorted_active_layers)}
    plot_height = len(sorted_active_layers)

    # Spans actually used (src + dst)
    used_spans = set()
    for e in top_edges:
        used_spans.add(e.src_span_name)
        used_spans.add(e.dst_span_name)

    # Assign colors to spans without one
    color_idx = len(span_color_map)
    for span in sorted(used_spans):
        
        if span in span_color_map:
            continue
            
        
        display_name = span_rename_map.get(span, span)
        
        
        if display_name in span_color_map:
            span_color_map[span] = span_color_map[display_name]
            continue

        
        span_color_map[span] = colors_list[color_idx % len(colors_list)]
        color_idx += 1

    # Canvas
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    ax.set_facecolor('white')

    for i in range(plot_height):
        if i % 2 == 1:
            ax.axhspan(
                i - 0.5, i + 0.5,
                facecolor="#F2F2F2",
                zorder=0
            )

    # Draw edges low->high score
    max_score = max(e.score for e in top_edges)
    for edge in top_edges[::-1]:
        if edge.src_span_name not in all_spans or edge.dst_span_name not in all_spans:
            continue

        src_x = all_spans.index(edge.src_span_name)
        dst_x = all_spans.index(edge.dst_span_name)
        src_y = layer_map[edge.layer - 1]
        dst_y = layer_map[edge.layer]

        src_color = span_color_map[edge.src_span_name]
        dst_color = span_color_map[edge.dst_span_name]

        xs, ys = sigmoid_path(src_x, src_y, dst_x, dst_y)

        # halo
        ax.plot(xs, ys, c='white', linewidth=3.5, zorder=1, solid_joinstyle='round',solid_capstyle='round',rasterized=True)

        
        ax.plot(xs, ys, c=src_color, linewidth=2, alpha=1.0, zorder=2, solid_joinstyle='round',solid_capstyle='round',rasterized=True)

        
        ax.scatter(src_x, src_y, c=[src_color], s=50,
                   edgecolors='black', linewidth=1.0, zorder=3)

        
        ax.scatter(dst_x, dst_y, c=[dst_color], s=50,
                   edgecolors='black', linewidth=1.0, zorder=3)

    
    ax.set_yticks(range(plot_height))
    y_labels = []
    for i, layer in enumerate(sorted_active_layers):
        if layer == -1:
            label = "Emb"
        else:
            label = f"L{layer}"

        if i == 0 or i == plot_height - 1 or i % label_interval == 0:
            y_labels.append(label)
        else:
            y_labels.append("")

    ax.set_yticklabels(y_labels, fontsize=12, fontweight='bold')

    # X axis (supports display rename)
    display_spans = [span_rename_map.get(s, s) for s in all_spans]

    ax.set_xticks(range(len(all_spans)))
    ax.set_xticklabels(display_spans, rotation=40,
                       ha='right', fontsize=15.5,rotation_mode='anchor')

    # Highlight special spans
    special_spans = {"tool-definition", "user-query"}
    for tick_label, span in zip(ax.get_xticklabels(), all_spans):
        if span in special_spans and span in span_color_map:
            darker = deepen_color(span_color_map[span], 0.75, 1.3)
            tick_label.set_color(darker)
            tick_label.set_fontweight('bold')

    # Frame
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1.0)

    ax.set_xlim(-0.5, len(all_spans) - 0.5)
    ax.set_ylim(-0.5, plot_height - 0.5)

    ax.set_title(title, fontsize=21, fontweight='bold', pad=9)
    plt.tight_layout()

    plt.savefig(f"figs/mech/pathways/{title.lower().replace(" ", "_")}_{model_name}.pdf", bbox_inches="tight",dpi=600)
    print(f"Saved figure: figs/mech/pathways/{title.lower().replace(' ', '_')}_{model_name}.pdf")
    plt.show()


In [ ]:
shared_color_map = {
    "system-start": "#B883D4",
    "tool-definition": "#A1A9D0",
    "user-query": "#96CCCB",
    "user-end": "#F6CAE5",
    "response": "#9E9E9E",
    "assistant-start": "#9E9E9E",
    "output-instr": "#F0988C",
    "tool-instr": "#C4A5DE",
    "BOS": "#32B897",
    
}
# colors_list = [
#     "#9E9E9E", "#F0988C", "#B883D4",
#     "#A1A9D0", "#F6CAE5", "#96CCCB",
#     "#CFEAF1",
# ]

span_rename_map = {
    "system-declaration-open": "system-start",
    "meta-instruction":"tool-instr",
    "tool-declaration-open":"tool-start",
    "tool-declaration":"tool-start",
    "tool-declaration-close":"tool-end",
    "output-instruction":"output-instr",
    "system-declaration-close":"system-end",
    "user-declaration-open": "user-start",
    "user-declaration-close": "user-end",
    "assistant-declaration-open": "assistant-start",
    "assistant-thinking": "response",
}

visualize_custom_final(
    top_str_c_edges,
    span_types,
    k=int(model.cfg.n_heads * model.cfg.n_layers * 0.02),
    figsize=(5.4, 5),
    label_interval=1,
    title="Structural Pathways",
    span_color_map=shared_color_map,
    span_rename_map=span_rename_map
)

visualize_custom_final(
    top_sem_c_edges,
    span_types,
    k=int(model.cfg.n_heads * model.cfg.n_layers * 0.02),
    figsize=(5.4, 5),
    label_interval=1,
    title="Semantic Pathways",
    span_color_map=shared_color_map,
    span_rename_map=span_rename_map
)
